# PatchTST 机械臂扭矩预测训练

## 说明
本notebook使用PatchTST模型（ICLR 2023）进行机械臂扭矩曲线预测。

**核心技术**：
- **PatchTST**: 最先进的时间序列预测架构
- **Residual Learning**: 预测差分值而非绝对值
- **Channel Independence**: 每个信号独立建模

**数据格式**：
- 文件名: `processed_Data_a_b_open.csv`
- 列名: `Time(s), T, Fx, Fy, Fz`
- 预测目标: Fy (索引1)

**预期效果**：
- R² > 0.5 (vs LSTM的-0.05)
- MAPE < 30% (vs LSTM的659%)

## 1. 环境设置

In [ ]:
# 导入必要的库
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import display, HTML

# 设置中文字体和绘图风格
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 负号显示
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')

# 检查CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 下载和准备数据

从Google Drive下载数据集并解压到本地。

**数据信息**：
- 文件: `processed_data.zip`
- 大小: ~几十MB
- 包含: `processed_data/` 文件夹，内有所有CSV文件

**注意**: 如果数据已存在，会自动跳过下载步骤。

In [ ]:
import zipfile
import shutil

# Google Drive文件ID
FILE_ID = "15sE6_eYeEq4TzPp4knR2jXP20kJirmAv"
OUTPUT_ZIP = "processed_data.zip"
DATA_FOLDER = "processed_data"

# 检查数据是否已存在
if os.path.exists('./data') and len(os.listdir('./data')) > 0:
    csv_count = len([f for f in os.listdir('./data') if f.endswith('.csv')])
    print("✅ 数据已存在，跳过下载")
    print(f"数据目录: ./data/")
    print(f"CSV文件数: {csv_count}")
else:
    print("开始下载数据...")
    print("=" * 60)
    
    # 方法1: 使用gdown (推荐)
    try:
        import gdown
        url = f"https://drive.google.com/uc?id={FILE_ID}"
        gdown.download(url, OUTPUT_ZIP, quiet=False)
        print("\n✅ 下载完成！")
    except ImportError:
        print("gdown未安装，正在安装...")
        import subprocess
        subprocess.check_call(["pip", "install", "-q", "gdown"])
        import gdown
        url = f"https://drive.google.com/uc?id={FILE_ID}"
        gdown.download(url, OUTPUT_ZIP, quiet=False)
        print("\n✅ 下载完成！")
    except Exception as e:
        print(f"❌ gdown下载失败: {e}")
        print("\n请手动下载数据：")
        print(f"1. 访问: https://drive.google.com/file/d/{FILE_ID}/view")
        print(f"2. 下载 {OUTPUT_ZIP}")
        print(f"3. 将文件放到当前目录")
        raise
    
    # 解压数据
    print("\n解压数据...")
    try:
        with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✅ 解压完成！")
        
        # 移动文件到 ./data/ 目录
        if os.path.exists(DATA_FOLDER):
            # 创建data目录
            os.makedirs('./data', exist_ok=True)
            
            # 移动所有CSV文件
            csv_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith('.csv')]
            print(f"\n移动 {len(csv_files)} 个CSV文件到 ./data/ ...")
            
            for csv_file in csv_files:
                src = os.path.join(DATA_FOLDER, csv_file)
                dst = os.path.join('./data', csv_file)
                shutil.move(src, dst)
            
            # 删除临时文件夹
            shutil.rmtree(DATA_FOLDER)
            print("✅ 文件移动完成！")
        
        # 删除zip文件
        os.remove(OUTPUT_ZIP)
        print("✅ 清理临时文件完成！")
        
        csv_count = len([f for f in os.listdir('./data') if f.endswith('.csv')])
        print("\n" + "=" * 60)
        print("数据准备完成！")
        print(f"数据目录: ./data/")
        print(f"CSV文件数: {csv_count}")
        print("=" * 60)
        
    except Exception as e:
        print(f"❌ 解压失败: {e}")
        print("\n请手动解压：")
        print(f"1. 解压 {OUTPUT_ZIP}")
        print(f"2. 将 {DATA_FOLDER}/ 文件夹中的CSV文件移动到 ./data/")
        raise

In [ ]:
## 3. 配置参数

## 2. 配置参数

In [ ]:
## 4. 加载数据

使用专门为PatchTST设计的固定长度数据加载器。

**特点**：
- 固定输入/输出长度
- 保留Residual Learning（返回差分值）
- 按瓶子编号划分train/test（防止数据泄漏）

## 3. 加载数据

使用专门为PatchTST设计的固定长度数据加载器。

**特点**：
- 固定输入/输出长度
- 保留Residual Learning（返回差分值）
- 按瓶子编号划分train/test（防止数据泄漏）

In [ ]:
## 5. 创建PatchTST模型

PatchTST架构：
1. **Patch Embedding**: 将2000步切分成~123个patches
2. **Transformer Encoder**: 3层multi-head attention
3. **Channel Independence**: 每个信号独立处理
4. **Prediction Head**: 输出1000步预测

## 4. 创建PatchTST模型

PatchTST架构：
1. **Patch Embedding**: 将2000步切分成~123个patches
2. **Transformer Encoder**: 3层multi-head attention
3. **Channel Independence**: 每个信号独立处理
4. **Prediction Head**: 输出1000步预测

In [ ]:
## 6. 训练模型

训练过程：
- 优化器: Adam
- 学习率调度: ReduceLROnPlateau
- 早停: 15个epoch无改进则停止
- 保存: 保存最佳模型到 `models_patchtst/best_model.pth`

## 5. 训练模型

训练过程：
- 优化器: Adam
- 学习率调度: ReduceLROnPlateau
- 早停: 15个epoch无改进则停止
- 保存: 保存最佳模型到 `models_patchtst/best_model.pth`

In [ ]:
## 7. 训练曲线可视化

## 6. 训练曲线可视化

In [ ]:
## 8. 评估Baseline模型

**Baseline**: Persistence Model（持久化预测）
- 策略: 用输入序列的最后一个Fy值预测所有输出
- 目的: 提供最简单的对比基准

## 7. 评估Baseline模型

**Baseline**: Persistence Model（持久化预测）
- 策略: 用输入序列的最后一个Fy值预测所有输出
- 目的: 提供最简单的对比基准

In [ ]:
## 9. 评估PatchTST模型

加载最佳模型并评估性能。

**注意**: 评估时会自动将预测的差分值重建为绝对值。

## 8. 评估PatchTST模型

加载最佳模型并评估性能。

**注意**: 评估时会自动将预测的差分值重建为绝对值。

In [ ]:
## 10. 结果对比分析

对比Baseline和PatchTST的性能。

## 9. 结果对比分析

对比Baseline和PatchTST的性能。

In [ ]:
## 11. 结论

总结PatchTST的表现：

## 10. 结论

总结PatchTST的表现：

In [ ]:
## 12. (可选) 超参数调优建议

如果结果不理想，可以尝试以下调整：

## 11. (可选) 超参数调优建议

如果结果不理想，可以尝试以下调整：

In [ ]:
print("\n超参数调优建议：\n")

if r2_value < 0.3:
    print("1. 增大模型容量：")
    print("   - D_MODEL = 256 (从128增大)")
    print("   - N_HEADS = 16 (从8增大)")
    print("   - E_LAYERS = 4 (从3增大)")
    print("")
    print("2. 缩短预测长度：")
    print("   - PRED_LEN = 500 (从1000减小)")
    print("")
    print("3. 增加训练轮数：")
    print("   - EPOCHS = 200 (从100增大)")
    print("")
    print("4. 调整学习率：")
    print("   - LEARNING_RATE = 0.0001 (更小的学习率)")

elif r2_value < 0.5:
    print("1. 适当增大模型：")
    print("   - D_MODEL = 192")
    print("   - N_HEADS = 12")
    print("")
    print("2. 增加数据增强：")
    print("   - STEP_SIZE = 200 (从500减小，创建更多样本)")
    print("")
    print("3. 延长训练：")
    print("   - EPOCHS = 150")
    print("   - EARLY_STOPPING_PATIENCE = 20")

else:
    print("当前结果已经很好！如果想进一步提升：")
    print("")
    print("1. 模型集成：")
    print("   - 训练多个PatchTST模型，使用不同的随机种子")
    print("   - 对预测结果取平均")
    print("")
    print("2. 数据预处理优化：")
    print("   - 移除异常值")
    print("   - 使用更复杂的特征工程")
    print("")
    print("3. 超参数微调：")
    print("   - 使用网格搜索或贝叶斯优化")
    print("   - 尝试不同的patch_len和stride组合")